Libraries and tools

In [2]:
import pandas as pd
import numpy as np
from PIL import Image
from io import BytesIO           
import statistics
import time
import matplotlib.pyplot as plt

Φόρτωση αρχείου Data

In [3]:
parquet_file = r'C:\Users\user\Desktop\pythonikos\train-00000-of-00001.parquet'

df = pd.read_parquet(parquet_file, engine='auto')

parquet_file = r'C:\Users\user\Desktop\pythonikos\test-00000-of-00001.parquet'

dftest = pd.read_parquet(parquet_file, engine='auto')

df.head()

,image,label
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,5
1,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,0
2,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,4
3,"{'bytes': b""\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...",1
4,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,9


Μετατροπή byte σε vector

In [4]:
train_set_synolo = 60000
test_set_synolo = 10000

topiki_boithitiki_lista_train = []
labels_lista_train = []

for i in range (train_set_synolo):
    raw_bytes = df.iloc[i]['image']['bytes']

    img = Image.open(BytesIO(raw_bytes))
    img = img.convert('L')

    arr = np.array(img)
    vector = arr.reshape(-1)
    topiki_boithitiki_lista_train.append(vector)

    labels_lista_train.append(df.iloc[i]['label'])

X_train = np.array(topiki_boithitiki_lista_train)
X_train = X_train.astype('float32') / 255.0



topiki_boithitiki_lista_test = []
labels_lista_test = []

for i in range (test_set_synolo):
    raw_bytes = dftest.iloc[i]['image']['bytes']

    img = Image.open(BytesIO(raw_bytes))
    img = img.convert('L')

    arr = np.array(img)
    vector = arr.reshape(-1)
    topiki_boithitiki_lista_test.append(vector)

    labels_lista_test.append(df.iloc[i]['label'])

X_test = np.array(topiki_boithitiki_lista_train)
X_test = X_train.astype('float32') / 255.0

Συλλογή 1000 δειγμάτων ανα κατηγορία 0...9 αρα 10.000 δειγματα

In [5]:
lista_me_10_trains = []

topiki_boithitiki_lista_katigorias = []

topiki_boithitiki_lista_katigorias_gia_8 = []                                                                                             #για 1000 vs 50


j = 0                                                                                                                                           

while len(topiki_boithitiki_lista_katigorias_gia_8) < 1000:
    if 8 == labels_lista_train[j]:
        topiki_boithitiki_lista_katigorias_gia_8.append(X_train[j])
    j+=1

for i in range(10):

    if i != 8:
        topiki_boithitiki_lista_katigorias = []

        j = 0

        while len(topiki_boithitiki_lista_katigorias) < 50:
            if i == labels_lista_train[j]:
                topiki_boithitiki_lista_katigorias.append(X_train[j])
            j+=1
        lista_me_10_trains.append(topiki_boithitiki_lista_katigorias)
    else:                                                                                                                                    #για 1000 vs 50
        lista_me_10_trains.append(topiki_boithitiki_lista_katigorias_gia_8)


# print('lens',len(lista_me_10_trains))
# for i in range(10):
#     print(i, ' ',len(lista_me_10_trains[i]))

lista_me_10_trains_labels = []

for i in range(10):
    if i != 8:
        lista_me_10_trains_labels.extend([i] * 50)
    else:
        lista_me_10_trains_labels.extend([i] * 1000)
# print('lens',len(lista_me_10_trains_labels))

# print(lista_me_10_trains_labels[9990])

Ευρεση βαρων-φιλτρων συγγρισης με δομή 2 πίνακες Συγγρισεις[45][2] πχ Συγγρισεις[1][0 , 2] συγγρινο το 0 με το 2  και βάρη[45] πχ βάρη[1] είναι το βάρος της σύγγρησης 0 με 2

Συγγρίσεις

In [6]:
comparison = []

for i in range(10):
    for j in range(i,10):
        if i != j:
            comparison.append([i, j])

# print(comparison)
# print('megethos',len(comparison))

Προβλεψη-απόφαση

In [7]:
def apofasi(a):
    if a <= 0:
        return -1
    return 1

Shuffling ανακάτεμα των δεδομένων μάθησης για κλτρ αποτελέσματα υπάρχει πιθανότητα να χρειαστεί να χρησιμοποιήσω και Epochs

In [8]:
# def shuffling(a, b, )

Weights Perceptron One vs One 

In [9]:
weights = []

learning_rate = 0.0001

epochs = 20

istoria_lathwn_synolika = [0] * epochs                                                                                                               #gia tis kampyles


for i in range(45):
    topiki_boithitiki_barwn = np.zeros(784)

    a = comparison[i][0]
    b = comparison[i][1]                                                    #παιρνω τις 2 συγγρισεις

    istoria_lathwn = []                                                                                                                            #gia tis kampyles 

    for e in range(epochs):

        lathi_epoxis = 0                                                                                                                           #gia tis kampyles


        # if a != 8 and b !=8:                                                                                                                        #για 1000 vs 50
        eikones_a = lista_me_10_trains[a]
        eikones_b = lista_me_10_trains[b]                                       #παίρνω τις 1000 εικόνες από τα a και b που σιγκρίνω

        combined_a_and_b = np.array(eikones_a + eikones_b)                      #τα κάνω combine

        combined_a_and_b_labels = np.array([1] * 1000 + [-1] * 1000)            #βάζω σαν labels τα 1 (a) και -1 (b)

        anakatema = np.arange(2000)                                             #laxnoi για το πως να το ανακατέψω

        np.random.shuffle(anakatema)                                            #ανακατέβω το anakatema δλδ τα 0...1999 σε 2,1,3,55... κλπ

        combined_a_and_b_anakatemena = combined_a_and_b[anakatema]              #εφαρμώζω το ανακατεμα στα 2

        combined_a_and_b_labels_anakatemena = combined_a_and_b_labels[anakatema]

        for j in range(2000):
            sum = 0

            # for z in range(784):
            #     sum += combined_a_and_b_anakatemena[j][z] * topiki_boithitiki_barwn[z]                  #πολλαπλασιαζω τα βαρη με την εικονα και κάνω το άθροισμα όλου του πινακα 
            
            sum = np.dot(combined_a_and_b_anakatemena[j], topiki_boithitiki_barwn)
            
            if apofasi(sum) == -1 and combined_a_and_b_labels_anakatemena[j] == 1:                      #αν κάνει λαθως για 1 τοτε ανεβάζω όλες τις θέσεις που είναι η εικόνα λατα ένα
                for z in range(784):

                    lathi_epoxis +=1                                                                                                                    #gia tis kampyles

                    topiki_boithitiki_barwn[z] += combined_a_and_b_anakatemena[j][z] * learning_rate
            elif apofasi(sum) == 1 and combined_a_and_b_labels_anakatemena[j] == -1:
                for z in range(784):
                    lathi_epoxis +=1                                                                                                                    #gia tis kampyles

                    topiki_boithitiki_barwn[z] -=combined_a_and_b_anakatemena[j][z] * learning_rate
        
            # weights.append(topiki_boithitiki_barwn)                                                                                               SOS TO BGAZV EKTOS LOUPAS



        #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
        # # else:----------------------------------------------------------------------------------------------------------------------------------------------- για 1000 vs 50---------------------------------------------
        #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


        #     eikones_a = lista_me_10_trains[a]
        #     eikones_b = lista_me_10_trains[b]                                       #παίρνω τις 1000 εικόνες από τα a και b που σιγκρίνω

        #     combined_a_and_b = np.array(eikones_a + eikones_b)                      #τα κάνω combine
        #     if a == 8:
        #         combined_a_and_b_labels = np.array([1] * 1000 + [-1] * 50)            #βάζω σαν labels τα 1 (a) και -1 (b)
        #     else:
        #         combined_a_and_b_labels = np.array([1] * 50 + [-1] * 1000)            #βάζω σαν labels τα 1 (a) και -1 (b)


        #     anakatema = np.arange(1050)                                             #laxnoi για το πως να το ανακατέψω

        #     np.random.shuffle(anakatema)                                            #ανακατέβω το anakatema δλδ τα 0...1999 σε 2,1,3,55... κλπ

        #     combined_a_and_b_anakatemena = combined_a_and_b[anakatema]              #εφαρμώζω το ανακατεμα στα 2

        #     combined_a_and_b_labels_anakatemena = combined_a_and_b_labels[anakatema]

        #     for j in range(1050):
        #         sum = 0

        #         for z in range(784):
        #             sum += combined_a_and_b_anakatemena[j][z] * topiki_boithitiki_barwn[z]                  #πολλαπλασιαζω τα βαρη με την εικονα και κάνω το άθροισμα όλου του πινακα 
                
        #         if apofasi(sum) == -1 and combined_a_and_b_labels_anakatemena[j] == 1:                      #αν κάνει λαθως για 1 τοτε ανεβάζω όλες τις θέσεις που είναι η εικόνα λατα ένα

        #             # lathi_epoxis +=1                                                                                                                    #gia tis kampyles

        #             for z in range(784):
        #                 topiki_boithitiki_barwn[z] += combined_a_and_b_anakatemena[j][z] * learning_rate
        #         elif apofasi(sum) == 1 and combined_a_and_b_labels_anakatemena[j] == -1:

        #             # lathi_epoxis +=1                                                                                                                    #gia tis kampyles

        #             for z in range(784):
        #                 topiki_boithitiki_barwn[z] -=combined_a_and_b_anakatemena[j][z] * learning_rate
        istoria_lathwn.append(lathi_epoxis)                                                                                                             #gia tis kampyles
        istoria_lathwn_synolika[e] += lathi_epoxis
    weights.append(topiki_boithitiki_barwn)



# plt.figure(figsize=(10, 6))                                                                                                                                 #gia tis kampyles
# # Βάζουμε τον συνολικό μας κουμπαρά στον άξονα Υ
# plt.plot(range(1, epochs + 1), istoria_lathwn_synolika, marker='o', color='purple')
# plt.title(f"Συνολική Καμπύλη Εκπαίδευσης και για τους 45 Ταξινομητές (LR: {learning_rate})")
# plt.xlabel("Εποχές (Epochs)")
# plt.ylabel("Συνολικά Λάθη (Από όλα τα ζευγάρια)")
# plt.grid(True)
# plt.show()

IndexError: index 1669 is out of bounds for axis 0 with size 100

testing - ψηφοφορία 

In [ ]:
lista_gia_test = X_test[0:2000]                                                                 #ορίζω 2000 δειγματα για τεστ μπορεί να θέλει πιο λίγα

lista_gia_test_labels = labels_lista_test[0:2000]
#print(len(lista_gia_test))

swsta_paradeigmata = {}                                                                                                                                 #gia paradeigmata
lathws_paradeigmata = {}                                                                                                                                #gia paradeigmata

pinakas_syn = np.zeros((10,10), dtype=int)

sum_epityxiwn = 0

for deigma_test_idx in range(2000):
    topiki_boithitiki_psifwn = np.zeros(10)

    for weight_table_idx in range(45):
        sum = 0 

        
        sum = np.dot(weights[weight_table_idx], lista_gia_test[deigma_test_idx])

        if apofasi(sum) == 1:
            topiki_boithitiki_psifwn[comparison[weight_table_idx][0]] +=1
        else:
            topiki_boithitiki_psifwn[comparison[weight_table_idx][1]] +=1
    
    pragmatiko_label = lista_gia_test_labels[deigma_test_idx]

    provlepomeno_label = np.argmax(topiki_boithitiki_psifwn)

    pinakas_syn[pragmatiko_label, provlepomeno_label] += 1

    if pragmatiko_label == provlepomeno_label:
        sum_epityxiwn +=1
    
        if pragmatiko_label not in swsta_paradeigmata :                             #and deigma_test_idx > 78 gia kalitero berdema kai alla paradeigmata                                #gia paradeigmata
            swsta_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), provlepomeno_label)
    else:
        # pass
        lathws_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), provlepomeno_label)                                   #gia paradeigmata

print('epityxies', sum_epityxiwn)

# ==========================================                                                                                                              #gia tis paradeigmata
# 3. ΤΥΠΩΜΑ ΣΩΣΤΩΝ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in swsta_paradeigmata:
#         img, pred = swsta_paradeigmata[i] # Τώρα το ξεπακετάρισμα θα δουλέψει τέλεια!
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='green')
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Σωστά Παραδείγματα", fontsize=16)
# plt.show()

# # ==========================================
# # 4. ΤΥΠΩΜΑ ΛΑΘΟΣ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# # ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in lathws_paradeigmata:
#         img, pred = lathws_paradeigmata[i]
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='red') 
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Λάθος Παραδείγματα (Πρόσεξε τα 8άρια!)", fontsize=16)
# plt.show()                                                                                                                                                #gia tis paradeigmata





# df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))

# print("Πίνακας Σύγχυσης:")
# print(df_cm)


epityxies 1930


Weights Perceptron One vs All

In [ ]:
weights = []

learning_rate = 0.0001

epochs = 20

istoria_lathwn_synolika = [0] * epochs                                                                                                               #gia tis kampyles


for sygkritis in range(10):
    topiki_boithitiki_barwn = np.zeros(784)

    istoria_lathwn = []                                                                                                                            #gia tis kampyles 


    for e in range(epochs):
        
        lathi_epoxis = 0                                                                                                                           #gia tis kampyles

        temp_labels_sygkrisewn = []

        for i in range(10):
            if i == sygkritis and i == 8:
                temp_labels_sygkrisewn.extend([1] * 1000)
            elif i != sygkritis and i == 8:
                temp_labels_sygkrisewn.extend([-1] * 1000)                                                              
            elif i != sygkritis:
                temp_labels_sygkrisewn.extend([-1] * 50)
            else:
                temp_labels_sygkrisewn.extend([1] * 50)

        # print("Συνολικό μέγεθος λίστας:", len(temp_labels_sygkrisewn))
        # print("Πόσα 1 έχει:", temp_labels_sygkrisewn.count(1))
        # print("Πόσα -1 έχει:", temp_labels_sygkrisewn.count(-1))

        
        topiki_boithitiki_sygkrisewn = [] 

        for i in range(10):
            topiki_boithitiki_sygkrisewn.extend(lista_me_10_trains[i])
                
        sygkriseis = np.array(topiki_boithitiki_sygkrisewn)

        # print(len(sygkriseis))

        labels_sygkrisewn = np.array(temp_labels_sygkrisewn)

        anakatema = np.arange(1450)

        np.random.shuffle(anakatema)

        sygkriseis_anakatemena = sygkriseis[anakatema]

        labels_sygkrisewn_anakatemena = labels_sygkrisewn[anakatema]

        for i in range(1450):

            sum = np.dot(sygkriseis_anakatemena[i], topiki_boithitiki_barwn)

            if apofasi(sum) == -1 and labels_sygkrisewn_anakatemena[i] == 1:
                lathi_epoxis +=1                                                                                                                    #gia tis kampyles

                topiki_boithitiki_barwn += sygkriseis_anakatemena[i] * learning_rate

            elif apofasi(sum) == 1 and labels_sygkrisewn_anakatemena[i] == -1:
                
                lathi_epoxis +=1                                                                                                                    #gia tis kampyles
                
                topiki_boithitiki_barwn -= sygkriseis_anakatemena[i] * learning_rate
    

        istoria_lathwn_synolika[e] += lathi_epoxis                                                                                                      #gia tis kampyles
    
    weights.append(topiki_boithitiki_barwn)


# plt.figure(figsize=(10, 6))
# plt.plot(range(1, epochs + 1), istoria_lathwn_synolika, marker='o', color='blue', linewidth=2)
# plt.title(f"Συνολική Καμπύλη Εκπαίδευσης One-vs-All (LR: {learning_rate})")
# plt.xlabel("Εποχές (Epochs)")
# plt.ylabel("Συνολικά Λάθη (Και οι 10 Ταξινομητές)")
# plt.grid(True)
# plt.show()





testing-ψηφοφορία

In [ ]:
lista_gia_test = X_test[0:2000]

lista_gia_test_labels = labels_lista_test[0:2000]

sum_epityxiwn = 0

pinakas_syn = np.zeros((10,10), dtype=int)                                                                              #gia pinaka

swsta_paradeigmata = {}                                                                                                                                 #gia paradeigmata
lathws_paradeigmata = {}                                                                                                                                #gia paradeigmata

for deigma_test_idx in range(2000):

    apotelesma_arithmos = 0

    max_score_arithmos = -float('inf')

    for baros_idx in range(10):
        sum = np.dot(lista_gia_test[deigma_test_idx], weights[baros_idx])
        if sum > max_score_arithmos:
            max_score_arithmos = sum
            apotelesma_arithmos = baros_idx
    
    pragmatiko_label = lista_gia_test_labels[deigma_test_idx]

    pinakas_syn[lista_gia_test_labels[deigma_test_idx], apotelesma_arithmos] += 1                                                                          #gia pinaka


    if pragmatiko_label == apotelesma_arithmos:
        sum_epityxiwn+=1

        if apotelesma_arithmos not in swsta_paradeigmata and deigma_test_idx > 300:
            swsta_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), apotelesma_arithmos)
    else:
        if apotelesma_arithmos not in lathws_paradeigmata and deigma_test_idx > 300:
            lathws_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), apotelesma_arithmos)



print(sum_epityxiwn)

# df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))                                                           #gia pinaka

# print("Πίνακας Σύγχυσης:")
# print(df_cm)                                                                                                                    #gia pinaka


# # ==========================================                                                                                                              #gia tis paradeigmata
# # 3. ΤΥΠΩΜΑ ΣΩΣΤΩΝ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# # ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in swsta_paradeigmata:
#         img, pred = swsta_paradeigmata[i] # Τώρα το ξεπακετάρισμα θα δουλέψει τέλεια!
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='green')
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Σωστά Παραδείγματα", fontsize=16)
# plt.show()

# # ==========================================
# # 4. ΤΥΠΩΜΑ ΛΑΘΟΣ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# # ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in lathws_paradeigmata:
#         img, pred = lathws_paradeigmata[i]
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='red') 
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Λάθος Παραδείγματα (Πρόσεξε τα 8άρια!)", fontsize=16)
# plt.show()                                                                                                                                                #gia tis paradeigmata

199


Softmax

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

Γραμμικός ταξινομητής με 10 εξόδους και softmax

In [ ]:
learning_rate = 0.0001

epochs = 20

alithia = []

for i in range(10):
    topiki_boithitiki_alithias = []
    for j in range(10):
        if i == j:
            topiki_boithitiki_alithias.append(1)
        else:
            topiki_boithitiki_alithias.append(0)
    alithia.append(topiki_boithitiki_alithias)

alithia = np.array(alithia)

weights = np.zeros(7840).reshape(10, 784)                                           #(10, 784)

istoria_lathwn_synolika = [0] * epochs                                                                                                               #gia tis kampyles

for e in range(epochs):
    # arxiko_train_set_flat = np.array(lista_me_10_trains).reshape(10000,784)

    temp_eikones = []

    for i in range(10):
        temp_eikones.extend(lista_me_10_trains[i])
    
    arxiko_train_set_flat = np.array(temp_eikones)

    arxiko_labels_flat = np.array(lista_me_10_trains_labels)

    anakatema = np.arange(1450)

    np.random.shuffle(anakatema)

    train_set_flat = arxiko_train_set_flat[anakatema]

    labels_flat = arxiko_labels_flat[anakatema]

    lathi_epoxis = 0

    for deigma_train_idx in range(1450):

        apotelesma_weight = np.dot(weights, train_set_flat[deigma_train_idx])                       #weight me eikona pollaplasiazv apotelesma [x,x,x,x,x,x,x,x,x]

        trexwn_alitheia = alithia[labels_flat[deigma_train_idx]]                                    #pairnw to monadiko syggriti poy ftiaxnv twra

        apotelesma_weight = softmax(apotelesma_weight)                                              #pososta twn sta 100 [%,%,%,%,%,%,...]

        errors = apotelesma_weight - trexwn_alitheia                                                #erros poy tha bgoun gia to pou prepei na kanw metabasi [+0.10, +0.10, +0.10, +0.10, +0.10, +0.10, +0.10, -0.90, +0.10, +0.10]

        weights = weights - learning_rate * np.outer(errors, train_set_flat[deigma_train_idx])

        if np.argmax(apotelesma_weight) != labels_flat[deigma_train_idx]:
            lathi_epoxis += 1

    istoria_lathwn_synolika[e] += lathi_epoxis


# plt.figure(figsize=(10, 6))
# plt.plot(range(1, epochs + 1), istoria_lathwn_synolika, marker='o', color='blue', linewidth=2)
# plt.title(f"Συνολική Καμπύλη Εκπαίδευσης (LR: {learning_rate})")
# plt.xlabel("Εποχές (Epochs)")
# plt.ylabel("Συνολικά Λάθη (Και οι 10 Ταξινομητές)")
# plt.grid(True)
# plt.show()

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


NameError: name 'softmax' is not defined

Testing

In [252]:
lista_gia_test = X_test[0:2000]

lista_gia_test_labels = labels_lista_test[0:2000]

swsta_paradeigmata = {}                                                                                                                                 #gia paradeigmata
lathws_paradeigmata = {}                                                                                                                                #gia paradeigmata

pinakas_syn = np.zeros((10,10), dtype=int)                                                                              #gia pinaka

sum_epityxiwn = 0

for deigma_test_idx in range(2000):

    scores = np.dot(weights, lista_gia_test[deigma_test_idx])

    apotelesma_arithmos = np.argmax(scores)

    pinakas_syn[lista_gia_test_labels[deigma_test_idx], apotelesma_arithmos] += 1                                                                          #gia pinaka

    pragmatiko_label = lista_gia_test_labels[deigma_test_idx]

    if apotelesma_arithmos == pragmatiko_label:
        sum_epityxiwn +=1

        if pragmatiko_label not in swsta_paradeigmata and deigma_test_idx > 500:
            swsta_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), apotelesma_arithmos)
    else:
        if pragmatiko_label not in lathws_paradeigmata and deigma_test_idx > 500:
            lathws_paradeigmata[pragmatiko_label] = (lista_gia_test[deigma_test_idx].reshape(28, 28), apotelesma_arithmos)

print(sum_epityxiwn)

# df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))                                                           #gia pinaka

# print("Πίνακας Σύγχυσης:")
# print(df_cm)                                                                                                                    #gia pinaka


# # ==========================================                                                                                                              #gia tis paradeigmata
# # 3. ΤΥΠΩΜΑ ΣΩΣΤΩΝ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# # ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in swsta_paradeigmata:
#         img, pred = swsta_paradeigmata[i] # Τώρα το ξεπακετάρισμα θα δουλέψει τέλεια!
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='green')
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Σωστά Παραδείγματα", fontsize=16)
# plt.show()

# # ==========================================
# # 4. ΤΥΠΩΜΑ ΛΑΘΟΣ ΠΑΡΑΔΕΙΓΜΑΤΩΝ
# # ==========================================
# plt.figure(figsize=(12, 5))
# for i in range(10):
#     if i in lathws_paradeigmata:
#         img, pred = lathws_paradeigmata[i]
#         plt.subplot(2, 5, i+1)
#         plt.imshow(img, cmap='gray')
#         plt.title(f"Real: {i}, Pred: {int(pred)}", color='red') 
#         plt.axis('off')
# plt.suptitle("Χαρακτηριστικά Λάθος Παραδείγματα (Πρόσεξε τα 8άρια!)", fontsize=16)
# plt.show()                                                                                                                                                #gia tis paradeigmata

555
